# 🧠 Transformer 모델 완전 실습

**모든 LLM의 기반이 되는 핵심 아키텍처를 코드로 직접 구현합니다.**

> "Attention Is All You Need" (Vaswani et al., 2017)

---

## 📋 실습 목차

| 섹션 | 내용 | 핵심 개념 |
|------|------|----------|
| 1 | 환경 설정 | PyTorch, 시각화 |
| 2 | Token Embedding | 토큰 → 벡터 변환 |
| 3 | Positional Encoding | 위치 정보 부여 (sin/cos) |
| 4 | Scaled Dot-Product Attention | Q·K^T / √d_k |
| 5 | Multi-Head Attention | 다중 헤드 병렬 처리 |
| 6 | Feed-Forward Network | Position-wise FFN |
| 7 | Encoder Layer | Self-Attention + FFN + Norm |
| 8 | Decoder Layer | Masked + Cross Attention |
| 9 | Full Transformer | 전체 모델 조립 |
| 10 | 번역 실습 | 한→영 간단 번역 학습 |
| 11 | Attention 시각화 | 어텐션 패턴 분석 |
| 12 | 현대 LLM 개선사항 | RoPE, RMSNorm, GQA |

---
**🔧 런타임 설정**: `런타임 > 런타임 유형 변경 > GPU(T4)` 권장

---
## 1. 환경 설정

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import rcParams
import seaborn as sns
import copy
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Colab 환경)
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
fe = fm.FontEntry(fname='/usr/share/fonts/truetype/nanum/NanumGothic.ttf', name='NanumGothic')
fm.fontManager.ttflist.insert(0, fe)
rcParams['font.family'] = fe.name
rcParams['axes.unicode_minus'] = False

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ PyTorch 버전: {torch.__version__}')
print(f'✅ 디바이스: {device}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

# 시드 고정
torch.manual_seed(42)
np.random.seed(42)

---
## 2. Token Embedding

각 토큰(단어)을 `d_model` 차원의 벡터로 변환합니다.

- Vocabulary에서 각 단어에 고유 인덱스 부여
- 인덱스 → 학습 가능한 임베딩 벡터로 매핑
- 원본 논문에서는 임베딩에 `√d_model`을 곱합니다

In [ ]:
class TokenEmbedding(nn.Module):
    """토큰 임베딩 레이어

    각 토큰 인덱스를 d_model 차원 벡터로 변환합니다.
    원본 논문에 따라 √d_model을 곱합니다.
    """
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x):
        # √d_model 스케일링: 임베딩 값이 Positional Encoding과 비슷한 크기가 되도록
        return self.embedding(x) * math.sqrt(self.d_model)


# === 실습: 토큰 임베딩 동작 확인 ===
vocab_size = 100  # 어휘 크기
d_model = 512     # 임베딩 차원 (원본 Transformer)

embed = TokenEmbedding(vocab_size, d_model)

# 예시: "나는 학교에 간다" → 토큰 인덱스
sample_tokens = torch.tensor([[10, 25, 42, 7]])  # (batch=1, seq_len=4)
embedded = embed(sample_tokens)

print(f'입력 형태:  {sample_tokens.shape}  → (batch, seq_len)')
print(f'출력 형태:  {embedded.shape}  → (batch, seq_len, d_model)')
print(f'\n첫 번째 토큰 벡터 (처음 10차원):')
print(embedded[0, 0, :10].detach().numpy().round(3))

In [ ]:
# === 시각화: 임베딩 벡터 히트맵 ===
fig, ax = plt.subplots(figsize=(12, 3))
sns.heatmap(
    embedded[0].detach().numpy()[:, :50],  # 처음 50차원만 표시
    cmap='RdBu_r', center=0, ax=ax,
    xticklabels=False,
    yticklabels=['토큰1 (나는)', '토큰2 (학교에)', '토큰3 (간다)', '토큰4 (EOS)']
)
ax.set_title('Token Embedding 벡터 (처음 50차원)', fontsize=14, fontweight='bold')
ax.set_xlabel('임베딩 차원', fontsize=11)
plt.tight_layout()
plt.show()
print('💡 각 토큰이 고유한 벡터 패턴을 가집니다.')

---
## 3. Positional Encoding

Transformer는 순서 정보가 없으므로, **위치 인코딩**을 추가합니다.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

- `pos`: 시퀀스 내 위치 (0, 1, 2, ...)
- `i`: 차원 인덱스
- 주기적 함수로 상대적 위치 관계를 표현

In [ ]:
class PositionalEncoding(nn.Module):
    """사인/코사인 기반 위치 인코딩

    학습하지 않는(fixed) 위치 정보를 임베딩에 더합니다.
    """
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # 위치 인코딩 행렬 계산
        pe = torch.zeros(max_len, d_model)           # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)

        # 10000^(2i/d_model) 의 역수를 로그 스케일로 계산
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)  # 짝수 인덱스: sin
        pe[:, 1::2] = torch.cos(position * div_term)  # 홀수 인덱스: cos

        pe = pe.unsqueeze(0)  # (1, max_len, d_model) → 배치 차원 추가
        self.register_buffer('pe', pe)  # 학습하지 않는 파라미터

    def forward(self, x):
        """x: (batch, seq_len, d_model)"""
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# === 실습: 위치 인코딩 생성 ===
pe_layer = PositionalEncoding(d_model=512, dropout=0.0)
pe_values = pe_layer.pe[0].numpy()  # (max_len, d_model)

print(f'위치 인코딩 형태: {pe_layer.pe.shape}')
print(f'위치 0, 차원 0 (sin): {pe_values[0, 0]:.4f}')
print(f'위치 0, 차원 1 (cos): {pe_values[0, 1]:.4f}')
print(f'위치 1, 차원 0 (sin): {pe_values[1, 0]:.4f}')

In [ ]:
# === 시각화: Positional Encoding 패턴 ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) 전체 히트맵
ax = axes[0]
im = ax.imshow(pe_values[:100, :64], cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_xlabel('임베딩 차원 (i)', fontsize=11)
ax.set_ylabel('위치 (pos)', fontsize=11)
ax.set_title('Positional Encoding 히트맵', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

# (2) 특정 차원의 사인/코사인 곡선
ax = axes[1]
positions = np.arange(100)
for dim_idx, label, color in [(0, 'dim 0 (sin)', '#3B82F6'),
                               (1, 'dim 1 (cos)', '#EF4444'),
                               (10, 'dim 10 (sin)', '#10B981'),
                               (50, 'dim 50 (sin)', '#F59E0B')]:
    ax.plot(positions, pe_values[:100, dim_idx], label=label, color=color, linewidth=1.5)
ax.set_xlabel('위치 (pos)', fontsize=11)
ax.set_ylabel('PE 값', fontsize=11)
ax.set_title('차원별 주기 변화', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('💡 낮은 차원 → 짧은 주기, 높은 차원 → 긴 주기')
print('💡 이를 통해 모델이 상대적 위치 관계를 학습할 수 있습니다.')

In [ ]:
# === 실습: 임베딩 + 위치 인코딩 결합 ===
pe_layer_with_drop = PositionalEncoding(d_model=512, dropout=0.1)

# Token Embedding 결과에 Positional Encoding 추가
final_input = pe_layer_with_drop(embedded)

print(f'Token Embedding 출력:     {embedded.shape}')
print(f'+ Positional Encoding 후: {final_input.shape}')
print(f'\n최종 입력 = Token Embedding + Positional Encoding')
print(f'(Dropout 0.1 적용 후 Encoder/Decoder에 입력됩니다)')

---
## 4. Scaled Dot-Product Attention

Transformer의 **핵심 연산**입니다.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

| 단계 | 연산 | 목적 |
|------|------|------|
| ① | Q·K^T | 토큰 간 유사도 계산 |
| ② | / √d_k | 그래디언트 안정화 |
| ③ | softmax | 확률 분포로 변환 |
| ④ | × V | 가중합으로 최종 표현 |


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled Dot-Product Attention

    Args:
        Q: Query  (batch, heads, seq_len, d_k)
        K: Key    (batch, heads, seq_len, d_k)
        V: Value  (batch, heads, seq_len, d_v)
        mask: 마스크 텐서 (optional)

    Returns:
        output: Attention 적용 결과
        attn_weights: Attention 가중치 (시각화용)
    """
    d_k = Q.size(-1)

    # ① Q·K^T : 유사도 점수 계산
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (batch, heads, seq_q, seq_k)
    print(f'  ① Q·K^T 점수 형태: {scores.shape}')

    # ② 스케일링: √d_k로 나누기
    scores = scores / math.sqrt(d_k)
    print(f'  ② 스케일링 후 (÷ √{d_k} = {math.sqrt(d_k):.2f})')

    # 마스크 적용 (Decoder의 미래 토큰 차단)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
        print(f'  🎭 마스크 적용됨')

    # ③ Softmax: 확률 분포로 변환
    attn_weights = F.softmax(scores, dim=-1)
    print(f'  ③ Softmax 후 가중치 합: {attn_weights[0,0,0].sum().item():.4f} (≈ 1.0)')

    # ④ V에 가중합 적용
    output = torch.matmul(attn_weights, V)
    print(f'  ④ 최종 출력 형태: {output.shape}')

    return output, attn_weights


# === 실습: Attention 계산 ===
batch_size = 1
n_heads = 1
seq_len = 4
d_k = 64

# 임의의 Q, K, V 생성
Q = torch.randn(batch_size, n_heads, seq_len, d_k)
K = torch.randn(batch_size, n_heads, seq_len, d_k)
V = torch.randn(batch_size, n_heads, seq_len, d_k)

print(f'Q 형태: {Q.shape}  (batch={batch_size}, heads={n_heads}, seq={seq_len}, d_k={d_k})')
print(f'K 형태: {K.shape}')
print(f'V 형태: {V.shape}')
print(f'\n--- Attention 계산 과정 ---')
output, attn_weights = scaled_dot_product_attention(Q, K, V)

In [ ]:
# === 시각화: Attention 가중치 히트맵 ===
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

tokens = ['나는', '학교에', '간다', '<EOS>']

# (1) 마스크 없는 Attention (Encoder 스타일)
ax = axes[0]
sns.heatmap(
    attn_weights[0, 0].detach().numpy(),
    annot=True, fmt='.3f', cmap='YlOrRd',
    xticklabels=tokens, yticklabels=tokens, ax=ax,
    vmin=0, vmax=0.6
)
ax.set_title('Encoder Self-Attention (마스크 없음)', fontsize=12, fontweight='bold')
ax.set_xlabel('Key (참조 대상)', fontsize=10)
ax.set_ylabel('Query (질의자)', fontsize=10)

# (2) Causal 마스크 Attention (Decoder 스타일)
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
_, masked_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

ax = axes[1]
sns.heatmap(
    masked_weights[0, 0].detach().numpy(),
    annot=True, fmt='.3f', cmap='YlOrRd',
    xticklabels=tokens, yticklabels=tokens, ax=ax,
    vmin=0, vmax=0.8
)
ax.set_title('Decoder Masked Self-Attention', fontsize=12, fontweight='bold')
ax.set_xlabel('Key (참조 대상)', fontsize=10)
ax.set_ylabel('Query (질의자)', fontsize=10)

plt.tight_layout()
plt.show()
print('💡 왼쪽: 모든 위치를 참조 (양방향)')
print('💡 오른쪽: 미래 토큰은 0 (삼각 마스크로 차단) → 자기회귀 생성')

---
## 5. Multi-Head Attention

하나의 Attention이 아닌, **여러 헤드**가 각기 다른 관점에서 정보를 추출합니다.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

- 원본: h=8 헤드, d_k = d_v = d_model / h = 64
- 각 헤드는 문법, 의미, 위치 등 다른 패턴을 학습

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention

    여러 개의 Attention 헤드를 병렬로 실행하고 결과를 결합합니다.
    """
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model은 n_heads로 나누어 떨어져야 합니다'

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # 각 헤드의 차원

        # Q, K, V 선형 변환 (하나의 행렬로 모든 헤드를 한번에 계산)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)  # 출력 프로젝션

        self.dropout = nn.Dropout(dropout)
        self.attn_weights = None  # 시각화용 저장

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # 1) 선형 변환 후 헤드 분리: (batch, seq, d_model) → (batch, heads, seq, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        # 2) Scaled Dot-Product Attention (각 헤드 병렬 처리)
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        self.attn_weights = attn_weights.detach()  # 시각화용 저장

        context = torch.matmul(attn_weights, V)

        # 3) 헤드 결합: (batch, heads, seq, d_k) → (batch, seq, d_model)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # 4) 출력 프로젝션
        output = self.W_o(context)

        return output


# === 실습: Multi-Head Attention ===
d_model = 512
n_heads = 8
mha = MultiHeadAttention(d_model, n_heads)

# 입력: (batch=2, seq_len=10, d_model=512)
x = torch.randn(2, 10, d_model)
output = mha(x, x, x)  # Self-Attention: Q=K=V=x

print(f'입력 형태:  {x.shape}')
print(f'출력 형태:  {output.shape}')
print(f'헤드 수:    {n_heads}')
print(f'헤드당 차원: {d_model // n_heads}')
print(f'\n파라미터 수: {sum(p.numel() for p in mha.parameters()):,}')
print(f'  - W_q: {d_model}×{d_model} = {d_model*d_model:,}')
print(f'  - W_k: {d_model}×{d_model} = {d_model*d_model:,}')
print(f'  - W_v: {d_model}×{d_model} = {d_model*d_model:,}')
print(f'  - W_o: {d_model}×{d_model} = {d_model*d_model:,}')

In [ ]:
# === 시각화: 8개 헤드의 Attention 패턴 ===
# 짧은 시퀀스로 패턴 확인
short_x = torch.randn(1, 6, d_model)
_ = mha(short_x, short_x, short_x)
weights = mha.attn_weights[0].numpy()  # (n_heads, 6, 6)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
labels = ['토큰1', '토큰2', '토큰3', '토큰4', '토큰5', '토큰6']

for idx, ax in enumerate(axes.flat):
    sns.heatmap(
        weights[idx], annot=True, fmt='.2f',
        cmap='Blues', vmin=0, vmax=0.5,
        xticklabels=labels, yticklabels=labels, ax=ax,
        cbar=False, annot_kws={'size': 8}
    )
    ax.set_title(f'Head {idx+1}', fontsize=11, fontweight='bold')
    ax.tick_params(labelsize=8)

plt.suptitle('Multi-Head Attention: 8개 헤드의 서로 다른 Attention 패턴',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('💡 각 헤드가 다른 패턴을 보입니다 → 다양한 관점의 정보 추출')

---
## 6. Feed-Forward Network & Layer Normalization

$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

- 두 개의 선형 변환 사이에 ReLU 활성화
- d_model(512) → d_ff(2048) → d_model(512)
- 각 위치에 독립적으로 동일하게 적용 (Position-wise)

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """Position-wise Feed-Forward Network

    FFN(x) = max(0, xW1 + b1)W2 + b2
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)    # 확장: 512 → 2048
        self.linear2 = nn.Linear(d_ff, d_model)    # 축소: 2048 → 512
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


class AddNorm(nn.Module):
    """Residual Connection + Layer Normalization

    Output = LayerNorm(x + SubLayer(x))
    """
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer_output):
        # Residual + Dropout + LayerNorm
        return self.norm(x + self.dropout(sublayer_output))


# === 실습: FFN 동작 확인 ===
d_model = 512
d_ff = 2048

ffn = PositionwiseFeedForward(d_model, d_ff)
add_norm = AddNorm(d_model)

x = torch.randn(1, 4, d_model)  # (batch, seq, d_model)
ffn_out = ffn(x)
normed = add_norm(x, ffn_out)

print(f'입력 형태:          {x.shape}')
print(f'FFN 출력:           {ffn_out.shape}')
print(f'Add & Norm 후:     {normed.shape}')
print(f'\nFFN 파라미터: {sum(p.numel() for p in ffn.parameters()):,}')
print(f'  - W1: {d_model}×{d_ff} = {d_model*d_ff:,}')
print(f'  - W2: {d_ff}×{d_model} = {d_ff*d_model:,}')
print(f'\n💡 FFN의 파라미터가 전체 레이어의 약 2/3를 차지합니다!')

In [ ]:
# === 시각화: Residual Connection 효과 ===
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

data_x = x[0, 0, :50].detach().numpy()
data_ffn = ffn_out[0, 0, :50].detach().numpy()
data_norm = normed[0, 0, :50].detach().numpy()

for ax, data, title, color in zip(axes,
    [data_x, data_ffn, data_norm],
    ['입력 (x)', 'FFN 출력', 'Add & Norm 결과'],
    ['#3B82F6', '#EF4444', '#10B981']):
    ax.bar(range(50), data, color=color, alpha=0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('차원', fontsize=10)
    ax.set_ylim(-4, 4)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)

plt.suptitle('Residual Connection + LayerNorm 효과 (처음 50차원)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 LayerNorm 후: 평균 ≈ 0, 표준편차 ≈ 1 로 안정화됩니다.')
print(f'   실제 평균: {normed[0,0].mean().item():.4f}, 표준편차: {normed[0,0].std().item():.4f}')

---
## 7. Encoder Layer

하나의 Encoder 레이어 = **Self-Attention → Add&Norm → FFN → Add&Norm**

원본 Transformer는 이 레이어를 **6개** 쌓습니다.

In [ ]:
class EncoderLayer(nn.Module):
    """Transformer Encoder 레이어 1개

    구성: Self-Attention → Add&Norm → FFN → Add&Norm
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        # Sub-layer 1: Multi-Head Self-Attention
        attn_output = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout1(attn_output))  # Residual + Norm

        # Sub-layer 2: Position-wise FFN
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_output))   # Residual + Norm

        return x


class Encoder(nn.Module):
    """Transformer Encoder: N개의 동일한 레이어 스택"""
    def __init__(self, d_model, n_heads, d_ff, n_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, src_mask=None):
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)


# === 실습: Encoder 구성 확인 ===
encoder = Encoder(d_model=512, n_heads=8, d_ff=2048, n_layers=6)

x = torch.randn(2, 10, 512)  # (batch=2, seq=10, d_model=512)
enc_output = encoder(x)

print(f'Encoder 입력: {x.shape}')
print(f'Encoder 출력: {enc_output.shape}')
print(f'\nEncoder 구조 (레이어 수: 6):')
total_params = sum(p.numel() for p in encoder.parameters())
print(f'총 파라미터: {total_params:,}')
print(f'\n각 레이어 구성:')
for name, param in list(encoder.layers[0].named_parameters())[:6]:
    print(f'  {name}: {param.shape}')

---
## 8. Decoder Layer

Decoder = **Masked Self-Attention → Add&Norm → Cross-Attention → Add&Norm → FFN → Add&Norm**

- **Masked Self-Attention**: 미래 토큰을 보지 못하도록 마스킹
- **Cross-Attention**: Encoder 출력(K,V)과 Decoder 현재 상태(Q)를 연결

In [ ]:
class DecoderLayer(nn.Module):
    """Transformer Decoder 레이어 1개

    구성: Masked Self-Attn → Add&Norm → Cross-Attn → Add&Norm → FFN → Add&Norm
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, tgt_mask=None, src_mask=None):
        # Sub-layer 1: Masked Multi-Head Self-Attention
        attn1 = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn1))

        # Sub-layer 2: Multi-Head Cross-Attention
        # Query = Decoder, Key/Value = Encoder 출력
        attn2 = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout2(attn2))

        # Sub-layer 3: Position-wise FFN
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_out))

        return x


class Decoder(nn.Module):
    """Transformer Decoder: N개의 동일한 레이어 스택"""
    def __init__(self, d_model, n_heads, d_ff, n_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, enc_output, tgt_mask=None, src_mask=None):
        for layer in self.layers:
            x = layer(x, enc_output, tgt_mask, src_mask)
        return self.norm(x)


# === 실습: Decoder 동작 확인 ===
decoder = Decoder(d_model=512, n_heads=8, d_ff=2048, n_layers=6)

# Causal 마스크 생성
tgt_len = 8
tgt_mask = torch.tril(torch.ones(tgt_len, tgt_len)).unsqueeze(0).unsqueeze(0)

tgt = torch.randn(2, tgt_len, 512)  # Decoder 입력
dec_output = decoder(tgt, enc_output, tgt_mask=tgt_mask)

print(f'Encoder 출력:  {enc_output.shape}')
print(f'Decoder 입력:  {tgt.shape}')
print(f'Causal 마스크: {tgt_mask.shape}')
print(f'Decoder 출력:  {dec_output.shape}')
print(f'\n총 파라미터:   {sum(p.numel() for p in decoder.parameters()):,}')

In [ ]:
# === 시각화: Causal Mask ===
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# (1) 마스크 패턴
ax = axes[0]
mask_vis = tgt_mask[0, 0].numpy()
sns.heatmap(mask_vis, annot=True, fmt='.0f', cmap='RdYlGn',
            xticklabels=[f't{i}' for i in range(tgt_len)],
            yticklabels=[f't{i}' for i in range(tgt_len)],
            ax=ax, cbar=False, vmin=0, vmax=1)
ax.set_title('Causal Mask (하삼각 행렬)', fontsize=12, fontweight='bold')
ax.set_xlabel('Key 위치', fontsize=10)
ax.set_ylabel('Query 위치', fontsize=10)

# (2) 마스크 적용 효과
ax = axes[1]
scores_before = torch.randn(tgt_len, tgt_len)
scores_after = scores_before.clone()
scores_after[tgt_mask[0, 0] == 0] = float('-inf')
probs = F.softmax(scores_after, dim=-1).numpy()
sns.heatmap(probs, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=[f't{i}' for i in range(tgt_len)],
            yticklabels=[f't{i}' for i in range(tgt_len)],
            ax=ax, cbar=True, vmin=0)
ax.set_title('마스크 적용 후 Softmax', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print('💡 1=참조 가능, 0=차단 → softmax 후 미래 위치는 확률 0')

---
## 9. Full Transformer 조립

모든 구성 요소를 결합하여 **완전한 Transformer 모델**을 구현합니다.

In [ ]:
class Transformer(nn.Module):
    """완전한 Transformer 모델

    구성: Embedding + PE → Encoder → Decoder → Linear → Softmax
    """
    def __init__(self, src_vocab, tgt_vocab, d_model=512, n_heads=8,
                 d_ff=2048, n_layers=6, dropout=0.1, max_len=5000):
        super().__init__()

        # 임베딩 레이어
        self.src_embed = TokenEmbedding(src_vocab, d_model)
        self.tgt_embed = TokenEmbedding(tgt_vocab, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)

        # Encoder & Decoder
        self.encoder = Encoder(d_model, n_heads, d_ff, n_layers, dropout)
        self.decoder = Decoder(d_model, n_heads, d_ff, n_layers, dropout)

        # 출력 레이어: d_model → target vocab
        self.output_linear = nn.Linear(d_model, tgt_vocab)

        # 파라미터 초기화 (Xavier uniform)
        self._init_parameters()

    def _init_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_causal_mask(self, size):
        """Decoder용 하삼각 마스크 생성"""
        mask = torch.tril(torch.ones(size, size)).unsqueeze(0).unsqueeze(0)
        return mask  # (1, 1, size, size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # 1) 임베딩 + 위치 인코딩
        src_emb = self.pos_enc(self.src_embed(src))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt))

        # 2) Decoder용 Causal Mask
        if tgt_mask is None:
            tgt_mask = self.make_causal_mask(tgt.size(1)).to(tgt.device)

        # 3) Encoder
        enc_output = self.encoder(src_emb, src_mask)

        # 4) Decoder
        dec_output = self.decoder(tgt_emb, enc_output, tgt_mask, src_mask)

        # 5) 출력 프로젝션 (logits)
        logits = self.output_linear(dec_output)

        return logits


# === 실습: 전체 Transformer 생성 ===
SRC_VOCAB = 1000   # 소스 어휘 크기
TGT_VOCAB = 1200   # 타겟 어휘 크기

model = Transformer(
    src_vocab=SRC_VOCAB,
    tgt_vocab=TGT_VOCAB,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    n_layers=6,
    dropout=0.1
)

# Forward pass 테스트
src = torch.randint(0, SRC_VOCAB, (2, 10))   # (batch=2, src_len=10)
tgt = torch.randint(0, TGT_VOCAB, (2, 8))    # (batch=2, tgt_len=8)

logits = model(src, tgt)

print('='*55)
print('       🏗️  Transformer 모델 구성 완료')
print('='*55)
print(f'소스 입력:    {src.shape}  → (batch, src_len)')
print(f'타겟 입력:    {tgt.shape}  → (batch, tgt_len)')
print(f'출력 logits:  {logits.shape}  → (batch, tgt_len, tgt_vocab)')
print(f'\n--- 하이퍼파라미터 (원본 논문) ---')
print(f'd_model:       512')
print(f'n_heads:       8')
print(f'd_ff:          2048')
print(f'n_layers:      6 (Encoder) + 6 (Decoder)')
print(f'd_k = d_v:     64 (= 512/8)')
total = sum(p.numel() for p in model.parameters())
print(f'\n총 파라미터:   {total:,} ({total/1e6:.1f}M)')

In [ ]:
# === 시각화: 모델 파라미터 분포 ===
components = {
    'Src Embedding': sum(p.numel() for p in model.src_embed.parameters()),
    'Tgt Embedding': sum(p.numel() for p in model.tgt_embed.parameters()),
    'Encoder (×6)': sum(p.numel() for p in model.encoder.parameters()),
    'Decoder (×6)': sum(p.numel() for p in model.decoder.parameters()),
    'Output Linear': sum(p.numel() for p in model.output_linear.parameters()),
}

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#3B82F6', '#8B5CF6', '#10B981', '#F59E0B', '#EF4444']
bars = ax.barh(list(components.keys()), list(components.values()), color=colors)

for bar, val in zip(bars, components.values()):
    ax.text(bar.get_width() + total*0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({val/total*100:.1f}%)', va='center', fontsize=10)

ax.set_xlabel('파라미터 수', fontsize=11)
ax.set_title(f'Transformer 파라미터 분포 (총 {total/1e6:.1f}M)', fontsize=13, fontweight='bold')
ax.set_xlim(0, max(components.values()) * 1.35)
plt.tight_layout()
plt.show()

---
## 10. 번역 실습: 간단한 한→영 번역

소규모 토이 데이터셋으로 Transformer의 **학습 과정**을 직접 확인합니다.

In [ ]:
# === 토이 데이터셋 준비 ===

# 간단한 한→영 번역 쌍
# 특수 토큰: <PAD>=0, <SOS>=1, <EOS>=2
PAD, SOS, EOS = 0, 1, 2

# 소스(한국어) 어휘
src_vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2,
             '나는': 3, '너는': 4, '그는': 5, '그녀는': 6,
             '학생': 7, '선생님': 8, '개발자': 9, '의사': 10,
             '이다': 11, '좋다': 12, '크다': 13,
             '매우': 14, '아주': 15}

# 타겟(영어) 어휘
tgt_vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2,
             'I': 3, 'you': 4, 'he': 5, 'she': 6,
             'am': 7, 'are': 8, 'is': 9,
             'a': 10, 'very': 11, 'really': 12,
             'student': 13, 'teacher': 14, 'developer': 15, 'doctor': 16,
             'good': 17, 'big': 18}

src_idx2word = {v: k for k, v in src_vocab.items()}
tgt_idx2word = {v: k for k, v in tgt_vocab.items()}

# 학습 데이터: (소스, 타겟) 쌍
train_pairs = [
    ([3, 7, 11, 2],       [1, 3, 7, 10, 13, 2]),         # 나는 학생 이다 → I am a student
    ([4, 8, 11, 2],       [1, 4, 8, 10, 14, 2]),         # 너는 선생님 이다 → you are a teacher
    ([5, 9, 11, 2],       [1, 5, 9, 10, 15, 2]),         # 그는 개발자 이다 → he is a developer
    ([6, 10, 11, 2],      [1, 6, 9, 10, 16, 2]),         # 그녀는 의사 이다 → she is a doctor
    ([3, 14, 12, 2],      [1, 3, 7, 11, 17, 2]),         # 나는 매우 좋다 → I am very good
    ([5, 15, 13, 2],      [1, 5, 9, 12, 18, 2]),         # 그는 아주 크다 → he is really big
    ([4, 14, 12, 2],      [1, 4, 8, 11, 17, 2]),         # 너는 매우 좋다 → you are very good
    ([6, 14, 13, 2],      [1, 6, 9, 11, 18, 2]),         # 그녀는 매우 크다 → she is very big
]

def pad_sequence(seq, max_len):
    return seq + [PAD] * (max_len - len(seq))

# 배치 텐서 생성
src_max = max(len(s) for s, _ in train_pairs)
tgt_max = max(len(t) for _, t in train_pairs)

src_tensor = torch.tensor([pad_sequence(s, src_max) for s, _ in train_pairs])
tgt_tensor = torch.tensor([pad_sequence(t, tgt_max) for _, t in train_pairs])

print(f'소스 어휘 크기: {len(src_vocab)}')
print(f'타겟 어휘 크기: {len(tgt_vocab)}')
print(f'학습 데이터: {len(train_pairs)}쌍')
print(f'\n소스 텐서: {src_tensor.shape}')
print(f'타겟 텐서: {tgt_tensor.shape}')
print(f'\n예시: {[src_idx2word[i.item()] for i in src_tensor[0]]}')
print(f'  →   {[tgt_idx2word[i.item()] for i in tgt_tensor[0]]}')

In [ ]:
# === 소규모 Transformer 학습 ===

# 작은 모델 (토이 데이터에 맞게)
small_model = Transformer(
    src_vocab=len(src_vocab),
    tgt_vocab=len(tgt_vocab),
    d_model=64,
    n_heads=4,
    d_ff=128,
    n_layers=2,
    dropout=0.1
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)  # PAD 무시
optimizer = optim.Adam(small_model.parameters(), lr=0.001, betas=(0.9, 0.98), eps=1e-9)

src_data = src_tensor.to(device)
tgt_data = tgt_tensor.to(device)

# 학습 루프
losses = []
n_epochs = 200

small_model.train()
for epoch in range(n_epochs):
    optimizer.zero_grad()

    # Teacher Forcing: 타겟의 마지막 토큰 제외 → 입력
    tgt_input = tgt_data[:, :-1]   # <SOS> I am a student
    tgt_label = tgt_data[:, 1:]    # I am a student <EOS>

    logits = small_model(src_data, tgt_input)

    # (batch*seq, vocab) 형태로 변환하여 Loss 계산
    loss = criterion(logits.reshape(-1, len(tgt_vocab)), tgt_label.reshape(-1))
    loss.backward()

    # 그래디언트 클리핑
    torch.nn.utils.clip_grad_norm_(small_model.parameters(), 1.0)
    optimizer.step()

    losses.append(loss.item())

    if (epoch + 1) % 40 == 0:
        print(f'Epoch [{epoch+1:3d}/{n_epochs}]  Loss: {loss.item():.4f}')

print(f'\n✅ 학습 완료! 최종 Loss: {losses[-1]:.4f}')

In [ ]:
# === 시각화: 학습 곡선 ===
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, color='#3B82F6', linewidth=1.5, alpha=0.8)
ax.fill_between(range(len(losses)), losses, alpha=0.1, color='#3B82F6')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Cross-Entropy Loss', fontsize=11)
ax.set_title('Transformer 학습 곡선', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

In [ ]:
# === 추론: Greedy Decoding ===

def greedy_decode(model, src, max_len=20):
    """Greedy Decoding으로 번역 수행"""
    model.eval()
    with torch.no_grad():
        # Encoder
        src_emb = model.pos_enc(model.src_embed(src))
        enc_output = model.encoder(src_emb)

        # Decoder: <SOS>부터 시작
        tgt_indices = [SOS]

        for _ in range(max_len):
            tgt_tensor = torch.tensor([tgt_indices]).to(src.device)
            tgt_emb = model.pos_enc(model.tgt_embed(tgt_tensor))
            tgt_mask = model.make_causal_mask(len(tgt_indices)).to(src.device)

            dec_output = model.decoder(tgt_emb, enc_output, tgt_mask)
            logits = model.output_linear(dec_output[:, -1, :])  # 마지막 위치
            next_token = logits.argmax(dim=-1).item()

            tgt_indices.append(next_token)
            if next_token == EOS:
                break

    return tgt_indices


# === 번역 테스트 ===
print('='*55)
print('       🔤  번역 결과 (Greedy Decoding)')
print('='*55)

for i, (src_seq, tgt_seq) in enumerate(train_pairs):
    src_input = torch.tensor([src_seq]).to(device)
    predicted = greedy_decode(small_model, src_input)

    src_text = ' '.join([src_idx2word[t] for t in src_seq if t not in [PAD, SOS, EOS]])
    tgt_text = ' '.join([tgt_idx2word[t] for t in tgt_seq if t not in [PAD, SOS, EOS]])
    pred_text = ' '.join([tgt_idx2word.get(t, '?') for t in predicted if t not in [PAD, SOS, EOS]])

    match = '✅' if pred_text == tgt_text else '❌'
    print(f'{match} [{src_text}] → 예측: [{pred_text}] (정답: [{tgt_text}])')

---
## 11. Attention 패턴 시각화

학습된 모델의 Attention 가중치를 시각화하여, 각 헤드가 어떤 패턴을 학습했는지 확인합니다.

In [ ]:
# === Attention 가중치 추출 ===

def get_attention_maps(model, src, tgt_input):
    """각 레이어의 Attention 가중치를 추출합니다."""
    model.eval()
    with torch.no_grad():
        src_emb = model.pos_enc(model.src_embed(src))
        tgt_emb = model.pos_enc(model.tgt_embed(tgt_input))

        # Encoder forward (Attention 저장)
        enc_x = src_emb
        enc_attns = []
        for layer in model.encoder.layers:
            enc_x = layer(enc_x)
            enc_attns.append(layer.self_attn.attn_weights)
        enc_x = model.encoder.norm(enc_x)

        # Decoder forward
        tgt_mask = model.make_causal_mask(tgt_input.size(1)).to(src.device)
        dec_x = tgt_emb
        dec_self_attns, dec_cross_attns = [], []
        for layer in model.decoder.layers:
            dec_x = layer(dec_x, enc_x, tgt_mask)
            dec_self_attns.append(layer.self_attn.attn_weights)
            dec_cross_attns.append(layer.cross_attn.attn_weights)

    return enc_attns, dec_self_attns, dec_cross_attns

# 첫 번째 예시로 Attention 추출
test_src = torch.tensor([train_pairs[0][0]]).to(device)     # 나는 학생 이다 <EOS>
test_tgt = torch.tensor([train_pairs[0][1][:-1]]).to(device) # <SOS> I am a student

enc_attns, dec_self_attns, dec_cross_attns = get_attention_maps(small_model, test_src, test_tgt)

print(f'Encoder Self-Attention: {len(enc_attns)} 레이어, 각 {enc_attns[0].shape}')
print(f'Decoder Self-Attention: {len(dec_self_attns)} 레이어')
print(f'Decoder Cross-Attention: {len(dec_cross_attns)} 레이어')

In [ ]:
# === 시각화: Encoder Self-Attention (레이어 1) ===
src_tokens = ['나는', '학생', '이다', '<EOS>']
tgt_tokens = ['<SOS>', 'I', 'am', 'a', 'student']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for h in range(4):
    ax = axes[h]
    w = enc_attns[0][0, h].cpu().numpy()
    sns.heatmap(w, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=0.6,
                xticklabels=src_tokens, yticklabels=src_tokens, ax=ax,
                cbar=False, annot_kws={'size': 9})
    ax.set_title(f'Head {h+1}', fontsize=11, fontweight='bold')

plt.suptitle('Encoder Self-Attention (Layer 1) — "나는 학생 이다"', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === 시각화: Decoder Cross-Attention (소스↔타겟 연결) ===
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for h in range(4):
    ax = axes[h]
    w = dec_cross_attns[0][0, h].cpu().numpy()
    sns.heatmap(w, annot=True, fmt='.2f', cmap='Oranges', vmin=0, vmax=0.6,
                xticklabels=src_tokens, yticklabels=tgt_tokens, ax=ax,
                cbar=False, annot_kws={'size': 9})
    ax.set_title(f'Head {h+1}', fontsize=11, fontweight='bold')

plt.suptitle('Decoder Cross-Attention (Layer 1) — 소스↔타겟 정렬', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Cross-Attention: 영어 토큰이 어떤 한국어 토큰을 참조하는지 보여줍니다.')
print('   예: "student" → "학생" 에 높은 attention → 번역 정렬 학습!')

---
## 12. 현대 LLM의 핵심 개선사항

원본 Transformer 이후 현대 LLM에 적용된 주요 기술들을 구현합니다.

In [ ]:
# === 12-1. RMSNorm (Root Mean Square Normalization) ===
# LLaMA, Gemma 등에서 LayerNorm 대신 사용

class RMSNorm(nn.Module):
    """RMSNorm: LayerNorm보다 효율적 (평균 계산 생략)

    RMSNorm(x) = x / RMS(x) * γ
    RMS(x) = √(mean(x²) + ε)
    """
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight

# 비교 실험
x = torch.randn(1, 10, 512)
ln = nn.LayerNorm(512)
rmsn = RMSNorm(512)

ln_out = ln(x)
rmsn_out = rmsn(x)

print('=== LayerNorm vs RMSNorm ===')
print(f'LayerNorm  - 평균: {ln_out.mean().item():.6f}, 표준편차: {ln_out.std().item():.4f}')
print(f'RMSNorm    - 평균: {rmsn_out.mean().item():.6f}, 표준편차: {rmsn_out.std().item():.4f}')
print(f'\n💡 RMSNorm은 평균(mean) 계산을 생략하여 약 10% 빠릅니다.')
print(f'   LLaMA, Gemma, Mistral 등 최신 모델에서 표준으로 사용됩니다.')

In [ ]:
# === 12-2. Rotary Position Embedding (RoPE) ===
# GPT-NeoX, LLaMA, Qwen 등에서 사용

class RotaryPositionEmbedding(nn.Module):
    """RoPE: 상대적 위치 정보를 회전 변환으로 인코딩

    - 절대 위치가 아닌 '상대적 거리'를 인코딩
    - 길이 일반화(length generalization) 성능 우수
    """
    def __init__(self, d_model, max_len=5000, base=10000):
        super().__init__()
        # θ_i = base^(-2i/d) for i in [0, d/2)
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)

        # 미리 계산
        t = torch.arange(max_len).float()
        freqs = torch.outer(t, inv_freq)      # (max_len, d/2)
        emb = torch.cat([freqs, freqs], dim=-1)  # (max_len, d)
        self.register_buffer('cos_cached', emb.cos())
        self.register_buffer('sin_cached', emb.sin())

    def _rotate_half(self, x):
        """벡터의 전반/후반을 교차하여 회전"""
        x1 = x[..., :x.shape[-1]//2]
        x2 = x[..., x.shape[-1]//2:]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, q, k, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)  # (1, 1, seq, d)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)

        q_rot = q * cos + self._rotate_half(q) * sin
        k_rot = k * cos + self._rotate_half(k) * sin
        return q_rot, k_rot

# RoPE 실습
rope = RotaryPositionEmbedding(d_model=64)
q = torch.randn(1, 4, 10, 64)  # (batch, heads, seq, d_k)
k = torch.randn(1, 4, 10, 64)
q_rot, k_rot = rope(q, k, seq_len=10)

print('=== Rotary Position Embedding (RoPE) ===')
print(f'Q 원본: {q.shape} → Q 회전: {q_rot.shape}')
print(f'K 원본: {k.shape} → K 회전: {k_rot.shape}')
print(f'\n💡 RoPE는 Q·K^T 내적 시 상대적 위치 (m-n) 만 반영됩니다.')
print(f'   → 학습 시 보지 못한 긴 시퀀스에도 일반화 가능!')

In [ ]:
# === 12-3. SwiGLU Activation (현대 FFN) ===
# LLaMA, PaLM, Gemma 등에서 ReLU 대신 사용

class SwiGLU(nn.Module):
    """SwiGLU: Gated Linear Unit with Swish

    SwiGLU(x) = (xW₁ · σ(xW₁)) ⊙ (xW_gate)
    - Swish 활성화 + Gate 메커니즘
    - ReLU 대비 더 나은 성능
    """
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w_gate(x))

# 비교: ReLU FFN vs SwiGLU
relu_ffn = PositionwiseFeedForward(512, 2048)
swiglu_ffn = SwiGLU(512, 2048)

x = torch.randn(1, 10, 512)
relu_out = relu_ffn(x)
swiglu_out = swiglu_ffn(x)

print('=== ReLU FFN vs SwiGLU ===')
print(f'ReLU FFN  파라미터: {sum(p.numel() for p in relu_ffn.parameters()):,}')
print(f'SwiGLU    파라미터: {sum(p.numel() for p in swiglu_ffn.parameters()):,}')
print(f'\n출력 형태: {relu_out.shape} (동일)')
print(f'\n💡 SwiGLU는 게이트 행렬(W_gate)이 추가되어 파라미터가 더 많지만,')
print(f'   같은 파라미터 수 대비 성능이 우수합니다 (PaLM 논문 결과).')

In [ ]:
# === 12-4. Grouped-Query Attention (GQA) ===
# LLaMA 2, Mistral 등에서 추론 효율성을 위해 사용

class GroupedQueryAttention(nn.Module):
    """Grouped-Query Attention (GQA)

    - Multi-Head Attention: 각 헤드마다 별도의 K, V
    - Multi-Query Attention (MQA): 모든 헤드가 K, V 공유
    - GQA: K, V를 '그룹'으로 공유 (MHA와 MQA의 중간)

    → KV 캐시 메모리 절약 + 추론 속도 향상
    """
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads  # 각 KV 그룹이 공유하는 Q 헤드 수
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)          # Q: 모든 헤드
        self.W_k = nn.Linear(d_model, self.n_kv_heads * self.d_k)  # K: KV 헤드만
        self.W_v = nn.Linear(d_model, self.n_kv_heads * self.d_k)  # V: KV 헤드만
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, S, _ = x.shape
        Q = self.W_q(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, S, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, S, self.n_kv_heads, self.d_k).transpose(1, 2)

        # KV 헤드를 Q 헤드 수에 맞게 반복
        K = K.repeat_interleave(self.n_rep, dim=1)
        V = V.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.W_o(out)

# 비교: MHA vs GQA vs MQA
x = torch.randn(1, 10, 512)

mha = MultiHeadAttention(512, n_heads=8)
gqa = GroupedQueryAttention(512, n_heads=8, n_kv_heads=2)  # 4개 Q가 1개 KV 공유
mqa = GroupedQueryAttention(512, n_heads=8, n_kv_heads=1)  # 모든 Q가 1개 KV 공유

print('=== Attention 변형 비교 ===')
print(f'{"방식":<25} {"Q 헤드":>8} {"KV 헤드":>8} {"KV 파라미터":>15}')
print('-' * 60)
print(f'{"MHA (원본)":<25} {"8":>8} {"8":>8} {sum(p.numel() for p in [mha.W_k.weight, mha.W_v.weight]):>15,}')
print(f'{"GQA (LLaMA 2)":<25} {"8":>8} {"2":>8} {sum(p.numel() for p in [gqa.W_k.weight, gqa.W_v.weight]):>15,}')
print(f'{"MQA (PaLM)":<25} {"8":>8} {"1":>8} {sum(p.numel() for p in [mqa.W_k.weight, mqa.W_v.weight]):>15,}')
print(f'\n💡 GQA는 KV 캐시를 75% 절약하면서 MHA에 가까운 성능을 유지합니다.')
print(f'   → 추론 시 메모리 & 속도 모두 개선!')

In [ ]:
# === 종합 비교 시각화 ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) Attention 방식별 KV 캐시 비교
ax = axes[0]
methods = ['MHA\n(원본)', 'GQA\n(LLaMA 2)', 'MQA\n(PaLM)']
kv_params = [524288, 131072, 65536]
colors = ['#3B82F6', '#10B981', '#F59E0B']
bars = ax.bar(methods, kv_params, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, kv_params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10000,
            f'{val:,}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('KV 파라미터 수', fontsize=11)
ax.set_title('Attention 방식별 KV 파라미터', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# (2) 원본 vs 현대 LLM 구성 비교
ax = axes[1]
categories = ['위치 인코딩', '정규화', '활성화 함수', 'Attention', 'Pre/Post Norm']
original = ['Sinusoidal', 'LayerNorm', 'ReLU', 'MHA', 'Post-Norm']
modern = ['RoPE', 'RMSNorm', 'SwiGLU', 'GQA', 'Pre-Norm']

y_pos = np.arange(len(categories))
ax.barh(y_pos + 0.2, [1]*5, height=0.35, color='#94A3B8', label='원본 (2017)', alpha=0.8)
ax.barh(y_pos - 0.2, [1]*5, height=0.35, color='#10B981', label='현대 LLM', alpha=0.8)

for i, (orig, mod) in enumerate(zip(original, modern)):
    ax.text(0.5, i + 0.2, orig, ha='center', va='center', fontsize=10, color='white', fontweight='bold')
    ax.text(0.5, i - 0.2, mod, ha='center', va='center', fontsize=10, color='white', fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(categories)
ax.set_xlim(0, 1)
ax.set_xticks([])
ax.set_title('원본 Transformer vs 현대 LLM', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

---
## 📝 핵심 요약

| 구성 요소 | 핵심 수식 | 역할 |
|-----------|-----------|------|
| **Token Embedding** | `Embed(x) × √d_model` | 토큰 → 벡터 변환 |
| **Positional Encoding** | `sin/cos(pos / 10000^(2i/d))` | 위치 정보 부여 |
| **Self-Attention** | `softmax(QK^T / √d_k) · V` | 토큰 간 관계 포착 |
| **Multi-Head** | `Concat(head₁,...,h) · Wₒ` | 다양한 관점 병렬 추출 |
| **FFN** | `max(0, xW₁+b₁)W₂+b₂` | 비선형 변환 |
| **Add & Norm** | `LayerNorm(x + SubLayer(x))` | 안정적 학습 |
| **Causal Mask** | 하삼각 행렬 | 미래 토큰 차단 |

### 현대 LLM의 진화

| 원본 (2017) | 현대 LLM | 장점 |
|-------------|---------|------|
| Sinusoidal PE | **RoPE** | 길이 일반화 |
| LayerNorm | **RMSNorm** | 연산 효율성 |
| ReLU FFN | **SwiGLU** | 표현력 향상 |
| MHA | **GQA** | 추론 메모리 절약 |
| Post-Norm | **Pre-Norm** | 학습 안정성 |

---
**🎓 다음 단계**: 이 코드를 기반으로 더 큰 데이터셋(예: IWSLT)에서 학습하거나, Hugging Face의 `transformers` 라이브러리로 사전학습된 모델을 활용해 보세요!